In [23]:

# Importando bibliotecas
import requests
import pandas as pd
import dotenv
import os
import json
from urllib.parse import urljoin
from datetime import datetime


# Carregando as variáveis do arquivo .env
dotenv.load_dotenv(override=True)


polotrial_url = os.getenv('POLOTRIAL_API_URL')
print(f" URL: {polotrial_url}")

# dados de sessão - Produção
api_username = os.getenv('POLOTRIAL_API_USERNAME')
api_password = os.getenv('POLOTRIAL_API_PASSWORD')


# dados de sessão - Playground
# api_username = os.getenv('POLOTRIAL_PLAYGROUND_USERNAME')
# api_password = os.getenv('POLOTRIAL_PLAYGROUND_PASSWORD')


# Iniciando a sessão
# Corpo do login a ser utilizado no acesso
payload = {
    "nome": api_username,
    "password":api_password
}

auth_headers = {
    "Content-Type": "application/json",
}
session_url = f"{polotrial_url}/sessions"
print(f"session: {session_url}")
response = requests.request("POST", session_url, json=payload, headers=auth_headers)


# Verificar a resposta
print(f"Status Code: {response.status_code}")
print(session_url)

#Recuperar o cookie de autenticação
auth_cookie = response.cookies.get("userId")
endpoint_headers={
        "cookie": f"userId={auth_cookie}"
    }
print({endpoint_headers['cookie']})

# Mensagem de sucesso na optenção do cookie
if endpoint_headers['cookie']:
    print("Autenticação bem-sucedida. Cookie obtido.")
    print(endpoint_headers)
    print(polotrial_url)
else:
    print("Falha na autenticação. Verifique suas credenciais.")

 URL: https://api.polotrial.com/beta
session: https://api.polotrial.com/beta/sessions
Status Code: 200
https://api.polotrial.com/beta/sessions
{'userId=s%3AWgnI2iuaJiuH2SpEe_C4kOC7DUgqU09F.dewXzvgKUk4sV5hLNmViNheUF8XPwEeNgY9Nj%2Bu66sk'}
Autenticação bem-sucedida. Cookie obtido.
{'cookie': 'userId=s%3AWgnI2iuaJiuH2SpEe_C4kOC7DUgqU09F.dewXzvgKUk4sV5hLNmViNheUF8XPwEeNgY9Nj%2Bu66sk'}
https://api.polotrial.com/beta


In [24]:
generica_url = f"{polotrial_url}/generica"
query_params = {
    "nested":"true"
    # "apelido_protocolo":"oxandrolona"
}
response = requests.request("GET", generica_url, headers=endpoint_headers, params=query_params)
print (generica_url)
generica = response.json()
if response.status_code == 200:
    print("Dados obtidos com sucesso:")
    print(json.dumps(generica, indent=4))
    generica_df = pd.DataFrame(generica)
else:
    print(f"Falha ao obter os dados. Status Code: {response.status_code}")


https://api.polotrial.com/beta/generica
Dados obtidos com sucesso:
[
    {
        "id": 9349,
        "co_generica": 34,
        "ds_descricao": "Alexion",
        "st_status": 0,
        "ordem": null,
        "co_externo": null,
        "cor_status": null,
        "user_id_log": null,
        "dados_generica": {
            "id": 34,
            "ds_descricao": "Patrocinadores"
        }
    },
    {
        "id": 9350,
        "co_generica": 34,
        "ds_descricao": "BCRI",
        "st_status": 0,
        "ordem": null,
        "co_externo": null,
        "cor_status": null,
        "user_id_log": null,
        "dados_generica": {
            "id": 34,
            "ds_descricao": "Patrocinadores"
        }
    },
    {
        "id": 9351,
        "co_generica": 3,
        "ds_descricao": "Controle de estoque",
        "st_status": 0,
        "ordem": null,
        "co_externo": null,
        "cor_status": null,
        "user_id_log": null,
        "dados_generica": {
           

In [25]:
colunas_a_extrair = [
    'dados_generica'
]

#extrair as informações dos dicionários nas colunas desejadas
for coluna in colunas_a_extrair:
    dados_extraidos = generica_df[coluna].apply(lambda x: x if isinstance(x, dict) else {})
    dados_expandido = pd.json_normalize(dados_extraidos)
    dados_expandido.columns = [f"{coluna}_{subcoluna}" for subcoluna in dados_expandido.columns]
    generica_df = pd.concat([generica_df, dados_expandido], axis=1)

In [26]:
#filtrar os status de protocolo
filtro = "Status protocolo"

filtro_generica_df = generica_df[generica_df['dados_generica_ds_descricao'] == filtro]

In [27]:
protocolo_url = f"{polotrial_url}/protocolo"
protocolos_ativos = [
    "161", #Aguardando iniciação
    "39", # Recrutamento aberto
    "9236", # Recrutamento aberto
    '9263', # Aguardando Ativação do Centro
    "393", # Aguardando Ativação do Centro
    '9294', # Fase Contratual
    '819', # Aguardando o Pacote Regulatório
    '739', # Aprovação Anvisa
    '6', # Aprovação Regulatória
    '912', # Em apreciação ética
    '9163'# Aprovado pelo CEP
    
    
    
]

df_protocolos = []

for co_status in protocolos_ativos:
    query_params ={
        "nested":"true",
        "status_protocolo": co_status
    }
    
    try:
        response = requests.request(
            "GET",
            protocolo_url,
            headers = endpoint_headers,
            params = query_params
        )
        response.raise_for_status()
        print(f"Status {co_status} - {response.status_code}")
        protocolo_json = response.json()
        if isinstance(protocolo_json, dict):
            protocolo_json = [protocolo_json]
        df_protocolos.append(pd.DataFrame(protocolo_json))
    
    except Exception as e:
        print(f"Erro ao buscar status {co_status}: {e}")

if df_protocolos:
    protocolos_dataframe = pd.concat(df_protocolos, ignore_index=True)
    print(f"DataFrame de protocolos criado com {len(protocolos_dataframe)} registros.")
else:
    protocolos_dataframe = pd.DataFrame()
    print("Nenhum dado de protocolo foi recuperado.")

Status 161 - 200
Status 39 - 200
Status 9236 - 200
Status 9263 - 200
Status 393 - 200
Status 9294 - 200
Status 819 - 200
Status 739 - 200
Status 6 - 200
Status 912 - 200
Status 9163 - 200
DataFrame de protocolos criado com 29 registros.


In [28]:
#selecionar colunas a se manter
colunas_desejadas = [
    'id',
    'apelido_protocolo',
    'dados_co_centro',
    'dados_pi',
    'dados_status_protocolo',
    'dados_local_atendimento',
    'dados_dificuldade_recrutamento',
    'doenca',
    'intervencao',
    'data_visita_selecao',
    'data_estimada_inicio',
    'data_finalizacao_esperada',
    'meta_inclusao',
    'meta_inclusao_atual',
    'data_inicio_recrutamento',
    'data_fim_recrutamento',
    'data_fim_protocolo',
    'data_siv',
    'participantes_incluidos',
    'falhas',
    'inclusoes',
]

protocolos_filtrados_df = protocolos_dataframe[colunas_desejadas]

In [29]:
colunas_a_extrair = [
    'dados_co_centro',
    'dados_pi',
    'dados_status_protocolo',
    'dados_local_atendimento',
    'dados_dificuldade_recrutamento',
    

]

#extrair as informações dos dicionários nas colunas desejadas
for coluna in colunas_a_extrair:
    dados_extraidos = protocolos_filtrados_df[coluna].apply(lambda x: x if isinstance(x, dict) else {})
    dados_expandido = pd.json_normalize(dados_extraidos)
    dados_expandido.columns = [f"{coluna}_{subcoluna}" for subcoluna in dados_expandido.columns]
    #excluir as colunas originais
    protocolos_filtrados_df = protocolos_filtrados_df.drop(columns=[coluna])
    protocolos_filtrados_df = pd.concat([protocolos_filtrados_df, dados_expandido], axis=1)


In [30]:
print(protocolos_filtrados_df.columns)

Index(['id', 'apelido_protocolo', 'doenca', 'intervencao',
       'data_visita_selecao', 'data_estimada_inicio',
       'data_finalizacao_esperada', 'meta_inclusao', 'meta_inclusao_atual',
       'data_inicio_recrutamento', 'data_fim_recrutamento',
       'data_fim_protocolo', 'data_siv', 'participantes_incluidos', 'falhas',
       'inclusoes', 'dados_co_centro_id', 'dados_co_centro_descricao',
       'dados_co_centro_co_externo', 'dados_pi_id', 'dados_pi_ds_nome',
       'dados_status_protocolo_id', 'dados_status_protocolo_ds_descricao',
       'dados_local_atendimento_id', 'dados_local_atendimento_ds_descricao',
       'dados_dificuldade_recrutamento_id',
       'dados_dificuldade_recrutamento_ds_descricao'],
      dtype='str')


In [31]:
#reordenar colunas na mesma sequencia da lista colunas desejadas
colunas_desejadas = [
    'id',
    'apelido_protocolo',
    'dados_co_centro_descricao',
    'dados_pi_ds_nome',
    'dados_status_protocolo_ds_descricao',
    'dados_local_atendimento_ds_descricao',
    'dados_dificuldade_recrutamento_ds_descricao',
    'doenca',
    'intervencao',
    'data_visita_selecao',
    'data_estimada_inicio',
    'data_finalizacao_esperada',
    'data_inicio_recrutamento',
    'data_fim_recrutamento',
    'data_fim_protocolo',
    'data_siv',
    'meta_inclusao',
    'meta_inclusao_atual',
    'participantes_incluidos',
    'falhas',
    'inclusoes',
]
protocolos_filtrados_df = protocolos_filtrados_df[colunas_desejadas]

In [32]:
#converter as datas para o formato DD-MM-AAAA
colunas_datas = [
    'data_visita_selecao',
    'data_estimada_inicio',
    'data_finalizacao_esperada',
    'data_inicio_recrutamento',
    'data_fim_recrutamento',
    'data_fim_protocolo',
    'data_siv'
]
for data in colunas_datas:
    protocolos_filtrados_df[data] = pd.to_datetime(protocolos_filtrados_df[data], errors='coerce').dt.strftime('%d-%m-%Y')
    

In [33]:
# Converter "meta_inclusao", "meta_inclusao_atual", "participantes_incluidos", "falhas" e "inclusoes" para inteiros
colunas_inteiros = [
    'meta_inclusao',
    'meta_inclusao_atual',
    'participantes_incluidos',
    'falhas',
    'inclusoes'
]
for coluna in colunas_inteiros:
    protocolos_filtrados_df[coluna] = pd.to_numeric(protocolos_filtrados_df[coluna], errors='coerce').fillna(0).astype(int)
    
#adicionar coluna de aproveitamento (inclusoes/participantes_incluidos)
protocolos_filtrados_df['taxa de não aproveitamento'] = protocolos_filtrados_df.apply(
    lambda row: (row['falhas'] / row['participantes_incluidos'] ) 
    if row['participantes_incluidos'] and row['participantes_incluidos'] > 0 else 0, axis=1
)
#Adicionar coluna de taxa de conclusão (participantes_incluidos/meta_inclusao_atual)
protocolos_filtrados_df['Randomizados/meta inicial'] = protocolos_filtrados_df.apply(
    lambda row: (row['inclusoes'] / row['meta_inclusao_atual'] ) 
    if row['meta_inclusao_atual'] and row['meta_inclusao_atual'] > 0 else 0, axis=1
)

In [34]:
pessoas_url = f"{polotrial_url}/pessoas"
query_params = {
    # "nested":"true"
    # "apelido_protocolo":"oxandrolona"
}
response = requests.request("GET", pessoas_url, headers=endpoint_headers, params=query_params)
print (pessoas_url)
pessoas = response.json()
if response.status_code == 200:
    print("Dados obtidos com sucesso:")
    # print(json.dumps(pessoas, indent=4))
    pessoas_df = pd.DataFrame(pessoas)
else:
    print(f"Falha ao obter os dados. Status Code: {response.status_code}")

https://api.polotrial.com/beta/pessoas
Dados obtidos com sucesso:


In [35]:
pessoas_df.head()

,dados_ativo,id,ds_nome,co_tipo_gn,co_equipe_gn,co_formacao_maior_nivel_gn,dt_ultimo_certificado_gcp,registro_profissional,declaracao_confidencialidade,custo_mensal_estimado,co_centro,equipe_padrao,equipe_padrao_kits,equipe_padrao_farmacia,equipe_padrao_naocega,filter,prox_vencimento_treinamento,co_especialidade,ativo,deleted
0,{'descricao': 'Sim'},2978,NaN,562.0,None,NaN,NaN,NaN,None,None,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0
1,{'descricao': 'Sim'},3545,NaN,562.0,None,NaN,NaN,NaN,None,None,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0
2,{'descricao': 'Sim'},1332,,649.0,None,NaN,NaN,NaN,None,None,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0
3,{'descricao': 'Sim'},3118,Alex Gonçalves Macedo,387.0,None,31.0,2023-09-06,74.334/SP,None,None,4.0,0,0.0,0.0,0.0,4.0,NaN,188.0,1,0
4,{'descricao': 'Sim'},1554,Alex Vladimir Dudzic,649.0,None,NaN,NaN,NaN,None,None,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0


In [36]:
# Extrair situação (dados_ativo) de pessoas_df
pessoas_df['dados_ativo_ds_descricao'] = pessoas_df['dados_ativo'].apply(lambda x: x.get('ds_descricao') if isinstance(x, dict) else None)
pessoas_df.head()

,dados_ativo,id,ds_nome,co_tipo_gn,co_equipe_gn,co_formacao_maior_nivel_gn,dt_ultimo_certificado_gcp,registro_profissional,declaracao_confidencialidade,custo_mensal_estimado,...,equipe_padrao,equipe_padrao_kits,equipe_padrao_farmacia,equipe_padrao_naocega,filter,prox_vencimento_treinamento,co_especialidade,ativo,deleted,dados_ativo_ds_descricao
0,{'descricao': 'Sim'},2978,NaN,562.0,None,NaN,NaN,NaN,None,None,...,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0,None
1,{'descricao': 'Sim'},3545,NaN,562.0,None,NaN,NaN,NaN,None,None,...,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0,None
2,{'descricao': 'Sim'},1332,,649.0,None,NaN,NaN,NaN,None,None,...,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0,None
3,{'descricao': 'Sim'},3118,Alex Gonçalves Macedo,387.0,None,31.0,2023-09-06,74.334/SP,None,None,...,0,0.0,0.0,0.0,4.0,NaN,188.0,1,0,None
4,{'descricao': 'Sim'},1554,Alex Vladimir Dudzic,649.0,None,NaN,NaN,NaN,None,None,...,0,0.0,0.0,0.0,NaN,NaN,NaN,1,0,None


In [37]:
import openpyxl

#exportar para excel
with pd.ExcelWriter(f'protocolos_filtrados_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx', engine='openpyxl') as writer:
    protocolos_filtrados_df.to_excel(writer, index=False, sheet_name='Protocolos Filtrados')

In [38]:
# # co_protocolo = protocolos_filtrados_df['id'].tolist()
# co_protocolo = [1182]
# print(co_protocolo)
# print(polotrial_url)
# participantes_url = f"{polotrial_url}/participantes"
# print(participantes_url)

# participantes_df_list = []
# for protocolo in co_protocolo:
#     query_params = {
#         "nested":"true",
#         "co_protocolo": protocolo
#     }
#     try:
#         response = requests.request(
#             "GET",
#             participantes_url,
#             headers = endpoint_headers,
#             params = query_params
#         )
#         response.raise_for_status()
        
#         data = response.json()
        
#         lista_final = []
        
#         if isinstance(data, list):
#             lista_final = data
#         elif isinstance(data, dict) and 'results' in data:
#             lista_final = data['results']
#         elif isinstance(data, dict):
#             lista_final = [data]
            
#         if lista_final:
#             participantes_df_list.append(pd.DataFrame(lista_final))
#             print(f"Protocolo {protocolo} - {response.status_code} - {len(lista_final)} registros recuperados.")
#         else:
#             print(f"Protocolo {protocolo} - {response.status_code} - Nenhum registro encontrado.")
#     except Exception as e:
#         print(f"Erro ao buscar participantes do protocolo {protocolo}: {e}")
        
# #Consolidação final
# if participantes_df_list:
#     participantes_dataframe = pd.concat(participantes_df_list, ignore_index=True)
#     print(f"DataFrame de participantes criado com {len(participantes_dataframe)} registros.")
# else:
#     participantes_dataframe = pd.DataFrame()
#     print("Nenhum dado de participante foi recuperado.")

In [39]:
# #filtrar as linhas onde dados_protocolo_apelido é igual a 'oxandrolona'
# oxandrolona_df = protocolo_procedimento_df[protocolo_procedimento_df['dados_protocolo_apelido_protocolo'].str.contains("oxandrolona", regex=False, na=False, case=False)]

In [40]:
# oxandrolona_df

In [41]:
# procedimentos_url = f"{polotrial_url}/procedimentos"
# query_params = {
#     "nested":"true",
#     # "apelido_protocolo":"oxandrolona"
# }
# response = requests.request("GET", procedimentos_url, headers=endpoint_headers, params=query_params)
# print(response.status_code)
# procedimentos = response.json()
# procedimentos_df = pd.DataFrame(procedimentos)

In [42]:
# procedimentos_df

In [54]:
participantes_url = f"{polotrial_url}/participantes"
query_params = {
    "nested":"true"
    # "apelido_protocolo":"oxandrolona"
}
response = requests.request("GET", participantes_url, headers=endpoint_headers, params=query_params)
print (participantes_url)
participantes = response.json()
if response.status_code == 200:
    print("Dados obtidos com sucesso:")
    # print(json.dumps(participantes, indent=4))
    participantes_df = pd.DataFrame(participantes)
else:
    print(f"Falha ao obter os dados. Status Code: {response.status_code}")

https://api.polotrial.com/beta/participantes
Dados obtidos com sucesso:


In [55]:
participantes_df.head()

,id,co_protocolo,id_participante,id_instituicao,nome,origem_indicacao,status_participante,iniciais,contatos,data_nascimento,...,co_status_gn,co_participante,dados_protocolo,dados_status,dados_braco,dados_voluntario,ParticipantesVisita,ParticipanteUltimaVisita,dados_proxima_visita,dados_equipe_protocolo
0,54,4,1001,5684766,Antonino Marmor Gomide,NaN,308,A.M.G,11999999999,1945-02-22,...,308,54,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 54, 'iniciais': 'A.M.G', 'ds_nome': 'An...","[{'id': 639, 'co_participante': 54, 'co_visita...","{'id': 54, 'co_voluntario': 54, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19..."
1,56,4,1002,5689356,Maria Pereira dos Santos,NaN,308,M.P.S,11999999999,1963-12-31,...,308,56,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 55, 'iniciais': 'M.P.S', 'ds_nome': 'Ma...","[{'id': 640, 'co_participante': 56, 'co_visita...","{'id': 56, 'co_voluntario': 55, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19..."
2,57,4,1003,5695907,Maria Sebastiana Babosa Vitor,NaN,310,M.S.V,11999999999,1957-01-20,...,310,57,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 310, 'ds_descricao': 'Falha de seleção'}","{'id': 1, 'nome': 'Único'}","{'id': 56, 'iniciais': 'M.S.V', 'ds_nome': 'Ma...","[{'id': 641, 'co_participante': 57, 'co_visita...","{'id': 57, 'co_voluntario': 56, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19..."
3,58,4,1004,5703918,Ronaldo Gomes da Silva,NaN,308,R.G.S,11999999999,1965-01-05,...,308,58,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 57, 'iniciais': 'R.G.S', 'ds_nome': 'Ro...","[{'id': 642, 'co_participante': 58, 'co_visita...","{'id': 58, 'co_voluntario': 57, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19..."
4,59,4,1005,5710802,Gilson Duarte Matioli,NaN,308,G.D.M,11999999999,1956-10-07,...,308,59,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 58, 'iniciais': 'G.D.M', 'ds_nome': 'Gi...","[{'id': 643, 'co_participante': 59, 'co_visita...","{'id': 59, 'co_voluntario': 58, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19..."


In [56]:
generica = generica_df.copy()
generica = generica[['id', 'ds_descricao']]
#renomear as colunas
generica = generica.rename(columns={'ds_descricao': 'generica_status'})

In [57]:
# merge entre df participantes e generica
participantes = participantes_df.merge(generica, left_on='status_participante', right_on='id', how='left')
participantes

,id_x,co_protocolo,id_participante,id_instituicao,nome,origem_indicacao,status_participante,iniciais,contatos,data_nascimento,...,dados_protocolo,dados_status,dados_braco,dados_voluntario,ParticipantesVisita,ParticipanteUltimaVisita,dados_proxima_visita,dados_equipe_protocolo,id_y,generica_status
0,54,4,1001,5684766,Antonino Marmor Gomide,NaN,308,A.M.G,11999999999,1945-02-22,...,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 54, 'iniciais': 'A.M.G', 'ds_nome': 'An...","[{'id': 639, 'co_participante': 54, 'co_visita...","{'id': 54, 'co_voluntario': 54, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19...",308,Concluído
1,56,4,1002,5689356,Maria Pereira dos Santos,NaN,308,M.P.S,11999999999,1963-12-31,...,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 55, 'iniciais': 'M.P.S', 'ds_nome': 'Ma...","[{'id': 640, 'co_participante': 56, 'co_visita...","{'id': 56, 'co_voluntario': 55, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19...",308,Concluído
2,57,4,1003,5695907,Maria Sebastiana Babosa Vitor,NaN,310,M.S.V,11999999999,1957-01-20,...,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 310, 'ds_descricao': 'Falha de seleção'}","{'id': 1, 'nome': 'Único'}","{'id': 56, 'iniciais': 'M.S.V', 'ds_nome': 'Ma...","[{'id': 641, 'co_participante': 57, 'co_visita...","{'id': 57, 'co_voluntario': 56, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19...",310,Falha de seleção
3,58,4,1004,5703918,Ronaldo Gomes da Silva,NaN,308,R.G.S,11999999999,1965-01-05,...,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 57, 'iniciais': 'R.G.S', 'ds_nome': 'Ro...","[{'id': 642, 'co_participante': 58, 'co_visita...","{'id': 58, 'co_voluntario': 57, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19...",308,Concluído
4,59,4,1005,5710802,Gilson Duarte Matioli,NaN,308,G.D.M,11999999999,1956-10-07,...,"{'id': 4, 'apelido_protocolo': 'BTK'}","{'id': 308, 'ds_descricao': 'Concluído'}","{'id': 1, 'nome': 'Único'}","{'id': 58, 'iniciais': 'G.D.M', 'ds_nome': 'Gi...","[{'id': 643, 'co_participante': 59, 'co_visita...","{'id': 59, 'co_voluntario': 58, 'ultima_visita...",None,"[{'id': 19, 'co_protocolo': 4, 'co_pessoa': 19...",308,Concluído
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,3334,2380,78-7,,Leandro Finotello Vicente,NaN,310,L.F.V,11969226083,1983-02-04,...,"{'id': 2380, 'apelido_protocolo': 'Oxandrolona'}","{'id': 310, 'ds_descricao': 'Falha de seleção'}","{'id': 253, 'nome': 'IMOX Versão nº1 - TRIAGEM'}","{'id': 3357, 'iniciais': 'L.F.V', 'ds_nome': '...","[{'id': 40809, 'co_participante': 3334, 'co_vi...","{'id': 3334, 'co_voluntario': 3357, 'ultima_vi...",None,"[{'id': 591413, 'co_protocolo': 2380, 'co_pess...",310,Falha de seleção
2478,3335,2380,78-5,,José Gustavo Targino de Oliveira,NaN,540,J.G.O,11992149823,1981-02-02,...,"{'id': 2380, 'apelido_protocolo': 'Oxandrolona'}","{'id': 540, 'ds_descricao': 'Triagem'}","{'id': 253, 'nome': 'IMOX Versão nº1 - TRIAGEM'}","{'id': 3359, 'iniciais': 'J.G.O', 'ds_nome': '...","[{'id': 40831, 'co_participante': 3335, 'co_vi...","{'id': 3335, 'co_voluntario': 3359, 'ultima_vi...",None,"[{'id': 591413, 'co_protocolo': 2380, 'co_pess...",540,Triagem
2479,3336,655,1076012035,1076012,Luiz Rodrigues Dourado,NaN,540,L.R.D,41996882884,1965-06-25,...,"{'id': 655, 'apelido_protocolo': 'EASi-HF - 13...","{'id': 540, 'ds_descricao': 'Triagem'}","{'id': 219, 'nome': 'Easi-HF 1378-0020'}","{'id': 3360, 'iniciais': 'L.R.D', 'ds_nome': '...","[{'id': 40834, 'co_participante': 3336, 'co_vi...","{'id': 3336, 'co_voluntario': 3360, 'ultima_vi...",None,"[{'id': 94694, 'co_protocolo': 655, 'co_pessoa...",540,Triagem
2480,3337,2230,3076012002,3076012,Claudio Nunes Vieira,NaN,540,C.N.V,4198923963,19

In [47]:
# filtrar os participantes com data_inclusão de 2025 em diante
participantes['data_inclusao'] = pd.to_datetime(participantes['data_inclusao'], errors='coerce')
participantes_2025 = participantes[participantes['data_inclusao'] >= '2025-01-01']
participantes_2025

,id_x,co_protocolo,id_participante,id_instituicao,nome,origem_indicacao,status_participante,iniciais,contatos,data_nascimento,...,atualizar_agenda,anexo_tcle,data_assinatura_tcle,data_da_falha,ultima_visita,proxima_visita,apagar_visitas_pendentes,dados_equipe_protocolo,id_y,generica_status
2141,2976,1182,0721-00161,0721,Valdete Tadim,NaN,11,V.-.T,41995615266,1965-01-01,...,1,anexos/svri_files/protocolos/1182/participante...,None,NaN,37985.0,37988.0,None,"[{'id': 373437, 'co_protocolo': 1182, 'co_pess...",11,Ativo
2142,2977,1182,0721-00162,0721,Gilmar Francisco Ribeiro,NaN,310,G.F.R,42998465476,1968-08-25,...,0,anexos/svri_files/protocolos/1182/participante...,None,2025-01-06T00:00:00.000Z,35209.0,NaN,None,"[{'id': 373437, 'co_protocolo': 1182, 'co_pess...",310,Falha de seleção
2143,2978,1182,0721-00163,0721,Slawomir Kopczynski,NaN,310,S.-.K,41998693795,1948-09-03,...,0,anexos/svri_files/protocolos/1182/participante...,None,2025-01-07T00:00:00.000Z,35217.0,NaN,None,"[{'id': 373437, 'co_protocolo': 1182, 'co_pess...",310,Falha de seleção
2145,2980,584,0761022-10003,10096377,Nicolas Martins de Goes,NaN,11,N.M.G,12992083702,2004-06-21,...,1,anexos/svri_files/protocolos/584/participantes...,None,NaN,40324.0,40373.0,None,"[{'id': 85561, 'co_protocolo': 584, 'co_pessoa...",11,Ativo
2146,2981,584,0761022-10004,10149860,Ivan Pedro Correa Nascimento Vieira,NaN,310,I.P.V,11941072890,2002-07-11,...,0,anexos/svri_files/protocolos/584/participantes...,None,2025-02-18T00:00:00.000Z,35684.0,NaN,None,"[{'id': 85561, 'co_protocolo': 584, 'co_pessoa...",310,Falha de seleção
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,3334,2380,78-7,,Leandro Finotello Vicente,NaN,310,L.F.V,11969226083,1983-02-04,...,1,NaN,None,2026-03-02T00:00:00.000Z,40809.0,40893.0,None,"[{'id': 591413, 'co_protocolo': 2380, 'co_pess...",310,Falha de seleção
2478,3335,2380,78-5,,José Gustavo Targino de Oliveira,NaN,540,J.G.O,11992149823,1981-02-02,...,1,NaN,None,NaN,40831.0,40833.0,None,"[{'id': 591413, 'co_protocolo': 2380, 'co_pess...",540,Triagem
2479,3336,655,1076012035,1076012,Luiz Rodrigues Dourado,NaN,540,L.R.D,41996882884,1965-06-25,...,1,anexos/svri_files/protocolos/655/participantes...,None,NaN,40834.0,40835.0,None,"[{'id': 94694, 'co_protocolo': 655, 'co_pessoa...",540,Triagem
2480,3337,2230,3076012002,3076012,Claudio Nunes Vieira,NaN,540,C.N.V,4198923963,1974-10-20,...,1,anexos/svri_files/protocolos/2230/participante...,None,NaN,40856.0,40857.0,None,"[{'id': 557210, 'co_protocolo': 2230, 'co_pess...",540,Triagem


In [48]:
# Contagem de participantes incluídos por status
contagem_status_inclusao_2025 = participantes_2025['generica_status'].value_counts()
print(contagem_status_inclusao_2025)

generica_status
Concluído               165
Falha de seleção         61
Ativo                    56
Triagem                  21
Falha de Pré Triagem     17
Descontinuado             5
Retirada de TCLE          5
Morte                     4
Perda de seguimento       3
EOT                       1
Name: count, dtype: int64


In [49]:
# Excluir linhas em que data_randomizacao seja nula ou em branco
participantes_2025 = participantes_2025.dropna(subset=['data_randomizacao'])
# Contagem de  participantes randomizados por status
contagem_randomizados_2025 = participantes_2025['generica_status'].value_counts()
print(contagem_randomizados_2025)

generica_status
Concluído              165
Ativo                   56
Retirada de TCLE         4
Perda de seguimento      3
Morte                    2
EOT                      1
Descontinuado            1
Triagem                  1
Name: count, dtype: int64


2026

In [50]:
# filtrar os participantes com data_inclusão de 2026 em diante
participantes['data_inclusao'] = pd.to_datetime(participantes['data_inclusao'], errors='coerce')
participantes_2026 = participantes[participantes['data_inclusao'] >= '2026-01-01']
participantes_2026

,id_x,co_protocolo,id_participante,id_instituicao,nome,origem_indicacao,status_participante,iniciais,contatos,data_nascimento,...,atualizar_agenda,anexo_tcle,data_assinatura_tcle,data_da_falha,ultima_visita,proxima_visita,apagar_visitas_pendentes,dados_equipe_protocolo,id_y,generica_status
2432,3284,1362,21209007,21209007,Isaura Maria dos Santos Cerqueira,NaN,11,I.S.C,(11) 94900-2025,1959-08-06,...,1,anexos/svri_files/protocolos/1362/participante...,None,NaN,40839.0,40840.0,None,"[{'id': 404978, 'co_protocolo': 1362, 'co_pess...",11,Ativo
2433,3285,1362,21209008,21209008,Pedro Vieira da Silva,NaN,310,P.V.S,+55 11 98762-9387,2006-11-15,...,1,anexos/svri_files/protocolos/1362/participante...,None,2026-02-09T00:00:00.000Z,40086.0,NaN,None,"[{'id': 404978, 'co_protocolo': 1362, 'co_pess...",310,Falha de seleção
2434,3286,784,5115001,5115001,Renato Bargieri França Freire,NaN,310,R.B.F,11994950809,1989-11-16,...,0,anexos/svri_files/protocolos/784/participantes...,None,2026-01-16T00:00:00.000Z,40105.0,NaN,None,"[{'id': 248967, 'co_protocolo': 784, 'co_pesso...",310,Falha de seleção
2435,3287,784,5115002,5115002,José Carlos Pereira,NaN,11,J.C.P,11918210627,1965-12-20,...,1,anexos/svri_files/protocolos/784/participantes...,None,NaN,40118.0,40119.0,None,"[{'id': 248967, 'co_protocolo': 784, 'co_pesso...",11,Ativo
2436,3288,1362,21209009,21209009,Alaide Lopes de Carvalho Fernandes,NaN,540,A.L.F,(11) 97793-3144,1995-02-23,...,1,anexos/svri_files/protocolos/1362/participante...,None,NaN,40127.0,NaN,None,"[{'id': 404978, 'co_protocolo': 1362, 'co_pess...",540,Triagem
2437,3289,1362,2109010,21209010,Enrique Gonzaga Carraro,NaN,540,E.G.C,(11) 975733777,2005-11-05,...,1,anexos/svri_files/protocolos/1362/participante...,None,NaN,40132.0,NaN,None,"[{'id': 404978, 'co_protocolo': 1362, 'co_pess...",540,Triagem
2438,3290,655,1076012028,1076012,Mirian Nazareth Fonseca,NaN,11,M.N.F,Amanda,1958-07-07,...,1,anexos/svri_files/protocolos/655/participantes...,None,NaN,40196.0,40197.0,None,"[{'id': 94694, 'co_protocolo': 655, 'co_pessoa...",11,Ativo
2439,3291,584,0761022-10014,0761022-10014,Raul Luiz Fernandes Ogliani,NaN,310,R.L.O,(11) 99407-2623,2005-03-05,...,0,anexos/svri_files/protocolos/584/participantes...,None,2026-02-26T00:00:00.000Z,40147.0,NaN,None,"[{'id': 85561, 'co_protocolo': 584, 'co_pessoa...",310,Falha de seleção
2440,3292,784,5115003,5115003,Liliam Reijani Silva,NaN,11,L.R.S,11993698593,2956-06-06,...,1,anexos/svri_files/protocolos/784/participantes...,None,NaN,40159.0,40160.0,None,"[{'id': 248967, 'co_protocolo': 784, 'co_pesso...",11,Ativo
2441,3296,584,0761022-10015,0761022-10015,Luciano de Marchi Neves,NaN,310,L.M.N,11 91519-6350,1987-10-19,...,0,anexos/svri_files/protocolos/584/participantes...,None,2026-02-12T00:00:00.000Z,40171.0,NaN,None,"[{'id': 85561, 'co_protocolo': 584, 'co_pessoa...",310,Falha de seleção


In [51]:
# Contagem de participantes incluídos por status
contagem_inclusoes_2026 = participantes_2026['generica_status'].value_counts()
print(contagem_inclusoes_2026)

generica_status
Triagem             21
Ativo               17
Falha de seleção    10
Name: count, dtype: int64


In [52]:
# Excluir linhas em que data_randomizacao seja nula ou em branco
participantes_2026 = participantes_2026.dropna(subset=['data_randomizacao'])
# Contagem de  participantes randomizados por status
contagem_randomizados_2026 = participantes_2026['generica_status'].value_counts()
print(contagem_randomizados_2026)

generica_status
Ativo      17
Triagem     1
Name: count, dtype: int64


Protocolos

In [58]:
# Participantes ativos totais (sem data de inclusão ou randomização), através de generica_status = "Ativo"
participantes_ativos = participantes[participantes['generica_status'] == 'Ativo']
contagem_participantes_ativos = participantes_ativos['generica_status'].value_counts()
print(contagem_participantes_ativos)


generica_status
Ativo    262
Name: count, dtype: int64


In [60]:
# Converter data_inclusao para datetime
participantes_ativos['data_inclusao'] = pd.to_datetime(participantes_ativos['data_inclusao'], errors='coerce')
#Classificar os participantes ativos por ano de inclusão
participantes_ativos['ano_inclusao'] = participantes_ativos['data_inclusao'].dt.year
contagem_ativos_por_ano = participantes_ativos['ano_inclusao'].value_counts().sort_index()
print(contagem_ativos_por_ano)

ano_inclusao
202       1
2022     39
2023     65
2024    101
2025     39
2026     17
Name: count, dtype: int64


In [53]:
protocolo_url = f"{polotrial_url}/protocolo"
query_params = {
    # "nested":"true"
    # "apelido_protocolo":"oxandrolona"
}
response = requests.request("GET", protocolo_url, headers=endpoint_headers, params=query_params)
print (protocolo_url)
protocolo = response.json()
if response.status_code == 200:
    print("Dados obtidos com sucesso:")
    # print(json.dumps(protocolo, indent=4))
    protocolo_df = pd.DataFrame(protocolo)
else:
    print(f"Falha ao obter os dados. Status Code: {response.status_code}")

https://api.polotrial.com/beta/protocolo
Dados obtidos com sucesso:
